# SegFormer - Efficient Semantic Segmentation with Transformers

**SegFormer** uses a hierarchical transformer encoder (MiT) + lightweight MLP decoder. More efficient than flat ViT-based SETR, often with better results.

Proposed by NVIDIA (NeurIPS 2021).  
Paper: [SegFormer: Simple and Efficient Design for Semantic Segmentation with Transformers](https://arxiv.org/abs/2105.15203)

**Architecture:**
- **Encoder**: MiT (Mix Transformer) - hierarchical, no positional encodings
- **Decoder**: Simple MLP that fuses multi-scale features

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import SegformerForSemanticSegmentation

# Reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# ============== CONFIG ==============
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 25
LR = 1e-4
NUM_CLASSES = 1

# SegFormer backbone: 'b0', 'b1', 'b2', 'b3', 'b4' (b0=lightest, b4=heaviest)
MODEL_SIZE = 'b0'
PRETRAINED_ID = f"nvidia/segformer-{MODEL_SIZE}-finetuned-ade-512-512"

BASE_PATH = "/run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task"
TRAIN_IMG = os.path.join(BASE_PATH, "train", "images")
TRAIN_MASK = os.path.join(BASE_PATH, "train", "masks")
TEST_IMG = os.path.join(BASE_PATH, "test", "images")
TEST_MASK = os.path.join(BASE_PATH, "test", "masks")

PRED_BASE = "/run/media/shaunakmishra25/DATA/CB/AI ML/BRISC/archive/brisc2025/segmentation_task/predictions/SegFormer"
PRED_MASK_DIR = os.path.join(PRED_BASE, "masks")
OVERLAY_DIR = os.path.join(PRED_BASE, "overlays")
MODEL_SAVE_DIR = os.path.join(os.path.dirname(PRED_BASE), "models")
MODEL_SAVE_PATH = os.path.join(MODEL_SAVE_DIR, f"segformer_{MODEL_SIZE}.pth")

os.makedirs(PRED_MASK_DIR, exist_ok=True)
os.makedirs(OVERLAY_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

## 1. Dataset & Preprocessing

SegFormer expects ImageNet normalization. For binary segmentation, `num_labels=1` uses BCE loss automatically.

In [ ]:
IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD)
])


class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.images = sorted(os.listdir(img_dir))
        self.masks = sorted(os.listdir(mask_dir))
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, self.masks[idx])

        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        mask = Image.open(mask_path).convert("L")
        mask = mask.resize((IMG_SIZE, IMG_SIZE))
        mask = np.array(mask).astype("float32") / 255.0
        mask = (mask > 0.5).astype("float32")
        mask = torch.tensor(mask)

        out_name = self.masks[idx]
        return image, mask, out_name


def collate_fn(batch):
    images = torch.stack([b[0] for b in batch])
    masks = torch.stack([b[1] for b in batch])
    names = [b[2] for b in batch]
    return images, masks, names

## 2. Model Setup

Load pre-trained SegFormer and replace decode head for binary segmentation (`num_labels=1` → BCEWithLogitsLoss).

In [ ]:
model = SegformerForSemanticSegmentation.from_pretrained(
    PRETRAINED_ID,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f"Device: {device}")
print(f"Backbone: {MODEL_SIZE}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
train_dataset = SegDataset(TRAIN_IMG, TRAIN_MASK, transform=train_transform)
test_dataset = SegDataset(TEST_IMG, TEST_MASK, transform=eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn
)

## 3. Training Loop

SegFormer outputs logits at reduced resolution; interpolate to mask size for loss.

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for images, masks, _ in loader:
        images = images.to(device)
        masks = masks.unsqueeze(1).to(device)
        optimizer.zero_grad()
        out = model(images)
        logits = out.logits
        logits = F.interpolate(logits, size=masks.shape[2:], mode='bilinear', align_corners=False)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, masks, _ in loader:
            images = images.to(device)
            masks = masks.unsqueeze(1).to(device)
            out = model(images)
            logits = out.logits
            logits = F.interpolate(logits, size=masks.shape[2:], mode='bilinear', align_corners=False)
            loss = criterion(logits, masks)
            total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)
    print(f"Epoch [{epoch}/{EPOCHS}] Loss: {train_loss:.4f} Val Loss: {val_loss:.4f}")

In [ ]:
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")

## 4. Inference & Visualization

In [ ]:
def denormalize(tensor):
    """Denormalize for overlay visualization."""
    mean = torch.tensor(IMAGE_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(IMAGE_STD).view(1, 3, 1, 1)
    return tensor * std + mean


model.eval()
torch.set_grad_enabled(False)

with torch.no_grad():
    for images, masks, names in test_loader:
        images = images.to(device)
        out = model(images)
        logits = out.logits
        logits = F.interpolate(logits, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        preds = torch.sigmoid(logits).cpu().numpy()
        preds_bin = (preds > 0.5).astype(np.uint8)

        for i, name in enumerate(names):
            pred_mask = preds_bin[i, 0]
            mask_out = Image.fromarray((pred_mask * 255).astype(np.uint8))
            mask_out.save(os.path.join(PRED_MASK_DIR, name))

            image_vis = denormalize(images[i : i + 1]).squeeze(0).permute(1, 2, 0).cpu().numpy()
            image_vis = np.clip(image_vis, 0, 1)
            overlay = image_vis.copy()
            overlay[pred_mask > 0] = [1, 0, 0]
            combined = 0.7 * image_vis + 0.3 * overlay
            combined = (np.clip(combined, 0, 1) * 255).astype(np.uint8)
            Image.fromarray(combined).save(os.path.join(OVERLAY_DIR, name))

print(f"Saved {len(test_dataset)} masks and overlays (filenames match ground truth)")


## 5. Test metrics & comparison report

Run **inference** above first so `masks/` and `overlays/` exist. If you skipped training, run **load checkpoint** then **re-run the inference cell**, then metrics and PDF.


In [ ]:
# Load saved .pth before inference if you skipped training (after model + DataLoader cells).
# Optional after a fresh train; use before metrics/report when reloading weights only.
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
model.eval()


In [ ]:
# Run after inference so PRED_MASK_DIR is populated.
def dice_np(y_true, y_pred, smooth=1e-6):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    intersection = np.sum(y_true * y_pred)
    return (2. * intersection + smooth) / (np.sum(y_true) + np.sum(y_pred) + smooth)

def iou_np(y_true, y_pred, smooth=1e-6):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    intersection = np.sum(y_true * y_pred)
    union = np.sum(y_true) + np.sum(y_pred) - intersection
    return (intersection + smooth) / (union + smooth)

def precision_recall_np(y_true, y_pred):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    precision = tp / (tp + fp + 1e-6)
    recall = tp / (tp + fn + 1e-6)
    return precision, recall

def accuracy_np(y_true, y_pred):
    y_true, y_pred = y_true.flatten(), y_pred.flatten()
    return np.sum(y_true == y_pred) / len(y_true)

dice_scores, iou_scores, precision_scores, recall_scores, accuracy_scores = [], [], [], [], []
for i in range(len(test_dataset)):
    _, mask, name = test_dataset[i]
    gt = (mask.numpy() > 0.5).astype(np.uint8)
    pred_path = os.path.join(PRED_MASK_DIR, name)
    pred_img = np.array(Image.open(pred_path))
    pred = (pred_img > 127).astype(np.uint8)
    dice_scores.append(dice_np(gt, pred))
    iou_scores.append(iou_np(gt, pred))
    p, r = precision_recall_np(gt, pred)
    precision_scores.append(p)
    recall_scores.append(r)
    accuracy_scores.append(accuracy_np(gt, pred))

print("===== TEST SET RESULTS =====")
print(f"Mean Dice     : {np.mean(dice_scores):.4f}")
print(f"Mean IoU      : {np.mean(iou_scores):.4f}")
print(f"Mean Precision: {np.mean(precision_scores):.4f}")
print(f"Mean Recall   : {np.mean(recall_scores):.4f}")
print(f"Mean Accuracy : {np.mean(accuracy_scores):.4f}")


In [ ]:
# Ground truth vs predicted comparison PDF (Image, GT, Predicted, Overlay, Comparison)
from matplotlib.backends.backend_pdf import PdfPages

SAMPLES_PER_PAGE = 4
COMPARISON_PDF_PATH = os.path.join(PRED_BASE, "comparison_report.pdf")


def make_comparison_overlay(gt_mask, pred_mask):
    """TP=green, FP=red, FN=blue."""
    gt = (gt_mask > 0.5).astype(np.uint8)
    pred = (pred_mask > 0.5).astype(np.uint8)
    h, w = gt.shape
    overlay = np.zeros((h, w, 3), dtype=np.uint8)
    overlay[gt == 1] = [0, 0, 128]
    overlay[pred == 1] = [128, 0, 0]
    overlay[(gt == 1) & (pred == 1)] = [0, 128, 0]
    return overlay


indices = np.arange(len(test_dataset))

with PdfPages(COMPARISON_PDF_PATH) as pdf:
    for page_start in range(0, len(indices), SAMPLES_PER_PAGE):
        page_indices = indices[page_start : page_start + SAMPLES_PER_PAGE]
        n_rows = len(page_indices)
        fig, axes = plt.subplots(n_rows, 5, figsize=(15, 3 * n_rows))
        if n_rows == 1:
            axes = axes.reshape(1, -1)
        for row, idx in enumerate(page_indices):
            img_name = test_dataset.images[idx]
            mask_name = test_dataset.masks[idx]
            img_path = os.path.join(TEST_IMG, img_name)
            gt_path = os.path.join(TEST_MASK, mask_name)
            pred_path = os.path.join(PRED_MASK_DIR, mask_name)
            overlay_path = os.path.join(OVERLAY_DIR, mask_name)

            image = np.array(Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))) / 255.0
            gt_mask = np.array(Image.open(gt_path).convert("L").resize((IMG_SIZE, IMG_SIZE))) / 255.0
            pred_mask = np.array(Image.open(pred_path).convert("L")) / 255.0
            if pred_mask.shape != (IMG_SIZE, IMG_SIZE):
                pred_mask = np.array(Image.fromarray((pred_mask * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))) / 255.0

            overlay_img = np.array(Image.open(overlay_path).convert("RGB")) / 255.0
            if overlay_img.shape[:2] != (IMG_SIZE, IMG_SIZE):
                overlay_img = np.array(Image.fromarray((overlay_img * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE))) / 255.0

            comparison = make_comparison_overlay(gt_mask, pred_mask)

            axes[row, 0].imshow(image)
            axes[row, 0].set_title("Image")
            axes[row, 0].axis("off")

            axes[row, 1].imshow(gt_mask, cmap="gray")
            axes[row, 1].set_title("Ground Truth")
            axes[row, 1].axis("off")

            axes[row, 2].imshow(pred_mask, cmap="gray")
            axes[row, 2].set_title("Predicted")
            axes[row, 2].axis("off")

            axes[row, 3].imshow(overlay_img)
            axes[row, 3].set_title("Overlay")
            axes[row, 3].axis("off")

            axes[row, 4].imshow(comparison)
            axes[row, 4].set_title("Comparison (TP=green, FP=red, FN=blue)")
            axes[row, 4].axis("off")

        plt.tight_layout()
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"Comparison PDF saved to {COMPARISON_PDF_PATH}")
